# Checkpoint Summary to One CSV

汇总 checkpoints_his 下所有 checkpoint 的参数与四个数据集平均指标（PSNR, MS-SSIM, SAM），并导出为单个 CSV。

In [1]:
import csv
import json
import math
import re
from pathlib import Path
from statistics import mean

CHECKPOINTS_ROOT = Path("checkpoints_his")
OUTPUT_CSV = Path("checkpoint_summary_all.csv")
OUTPUT_PER_IMAGE_CSV = Path("checkpoint_summary_per_image.csv")
TARGET_DATASETS = ["paviau", "salinas", "jasperridge", "urban"]

# Example folder name:
# rank10_lora3_a0.1_g1_pts5000_prune500_add5000_max10000_covariance
CKPT_NAME_RE = re.compile(
    r"^rank(?P<rank>\d+)_lora(?P<lora_rank>\d+)_a(?P<alpha>[0-9.]+)_g(?P<gabor>\d+)_"
    r"pts(?P<num_points>\d+)_prune(?P<prune_iter>\d+)_add(?P<add_iter>\d+)_max(?P<max_num_points>\d+)"
)

def parse_checkpoint_name(name: str):
    m = CKPT_NAME_RE.match(name)
    if not m:
        return {
            "Rank": "",
            "Lora_Rank": "",
            "Gabor": "",
            "Num_Points": "",
            "Prune_iter": "",
            "Add_iter": "",
            "Max_Num_Points": "",
        }

    d = m.groupdict()
    return {
        "Rank": int(d["rank"]),
        "Lora_Rank": int(d["lora_rank"]),
        "Gabor": int(d["gabor"]),
        "Num_Points": int(d["num_points"]),
        "Prune_iter": int(d["prune_iter"]),
        "Add_iter": int(d["add_iter"]),
        "Max_Num_Points": int(d["max_num_points"]),
    }

def is_valid_number(v):
    return isinstance(v, (int, float)) and not math.isnan(v)

def safe_float(v):
    return float(v) if is_valid_number(v) else ""

def safe_mean(values):
    vals = [v for v in values if is_valid_number(v)]
    if not vals:
        return ""
    return float(mean(vals))

def get_ms_ssim(metric: dict):
    for key in ["ms_ssim", "ms-ssim", "msssim", "MS_SSIM", "ms_ssim_value"]:
        if key in metric and is_valid_number(metric[key]):
            return float(metric[key])
    return None

rows = []
per_image_rows = []

for ckpt_dir in sorted(CHECKPOINTS_ROOT.iterdir()):
    if not ckpt_dir.is_dir():
        continue

    params = parse_checkpoint_name(ckpt_dir.name)

    psnr_vals = []
    sam_vals = []
    ms_ssim_vals = []
    found_datasets = []

    row = {
        "Checkpoint": ckpt_dir.name,
        "Num_Points": params["Num_Points"],
        "Max_Num_Points": params["Max_Num_Points"],
        "Rank": params["Rank"],
        "Lora_Rank": params["Lora_Rank"],
        "Gabor": params["Gabor"],
        "Prune_iter": params["Prune_iter"],
        "Add_iter": params["Add_iter"],
    }

    for ds in TARGET_DATASETS:
        row[f"{ds}_PSNR"] = ""
        row[f"{ds}_MS_SSIM"] = ""
        row[f"{ds}_SAM"] = ""

    for ds in TARGET_DATASETS:
        metric_path = ckpt_dir / ds / "metrics.json"
        if not metric_path.exists():
            continue

        metric = json.loads(metric_path.read_text(encoding="utf-8"))
        found_datasets.append(ds)

        psnr = safe_float(metric.get("psnr"))
        sam = safe_float(metric.get("sam"))
        ms_ssim_val = get_ms_ssim(metric)
        ms_ssim = safe_float(ms_ssim_val)

        row[f"{ds}_PSNR"] = psnr
        row[f"{ds}_MS_SSIM"] = ms_ssim
        row[f"{ds}_SAM"] = sam

        if is_valid_number(psnr):
            psnr_vals.append(psnr)
        if is_valid_number(sam):
            sam_vals.append(sam)
        if is_valid_number(ms_ssim):
            ms_ssim_vals.append(ms_ssim)

        per_image_rows.append(
            {
                "Checkpoint": ckpt_dir.name,
                "Dataset": ds,
                "Num_Points": params["Num_Points"],
                "Max_Num_Points": params["Max_Num_Points"],
                "Rank": params["Rank"],
                "Lora_Rank": params["Lora_Rank"],
                "Gabor": params["Gabor"],
                "Prune_iter": params["Prune_iter"],
                "Add_iter": params["Add_iter"],
                "PSNR": psnr,
                "MS_SSIM": ms_ssim,
                "SAM": sam,
            }
        )

    row.update(
        {
            "Avg_PSNR_4img": safe_mean(psnr_vals),
            "Avg_MS_SSIM_4img": safe_mean(ms_ssim_vals),
            "Avg_SAM_4img": safe_mean(sam_vals),
            "Found_Dataset_Count": len(found_datasets),
            "Found_Datasets": ",".join(found_datasets),
        }
    )
    rows.append(row)

def sort_key(r):
    def n(x):
        return x if isinstance(x, int) else 10**9

    return (n(r["Rank"]), n(r["Lora_Rank"]), n(r["Gabor"]), n(r["Num_Points"]))

rows = sorted(rows, key=sort_key)
per_image_rows = sorted(per_image_rows, key=lambda r: sort_key(r) + (r["Dataset"],))

fieldnames = [
    "Checkpoint",
    "Num_Points",
    "Max_Num_Points",
    "Rank",
    "Lora_Rank",
    "Gabor",
    "Prune_iter",
    "Add_iter",
]
for ds in TARGET_DATASETS:
    fieldnames.extend([f"{ds}_PSNR", f"{ds}_MS_SSIM", f"{ds}_SAM"])
fieldnames.extend(
    [
        "Avg_PSNR_4img",
        "Avg_MS_SSIM_4img",
        "Avg_SAM_4img",
        "Found_Dataset_Count",
        "Found_Datasets",
    ]
)

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

per_image_fieldnames = [
    "Checkpoint",
    "Dataset",
    "Num_Points",
    "Max_Num_Points",
    "Rank",
    "Lora_Rank",
    "Gabor",
    "Prune_iter",
    "Add_iter",
    "PSNR",
    "MS_SSIM",
    "SAM",
]

with open(OUTPUT_PER_IMAGE_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=per_image_fieldnames)
    writer.writeheader()
    writer.writerows(per_image_rows)

print(f"Saved {len(rows)} checkpoints to: {OUTPUT_CSV.resolve()}")
print(f"Saved {len(per_image_rows)} per-image rows to: {OUTPUT_PER_IMAGE_CSV.resolve()}")
print("Preview (first 3 rows):")
for r in rows[:3]:
    print(r)


Saved 96 checkpoints to: /data/lab/HSI_Gabor_plus/checkpoint_summary_all.csv
Saved 384 per-image rows to: /data/lab/HSI_Gabor_plus/checkpoint_summary_per_image.csv
Preview (first 3 rows):
{'Checkpoint': 'rank8_lora3_a0.1_g1_pts2500_prune500_add5000_max5000_covariance', 'Num_Points': 2500, 'Max_Num_Points': 5000, 'Rank': 8, 'Lora_Rank': 3, 'Gabor': 1, 'Prune_iter': 500, 'Add_iter': 5000, 'paviau_PSNR': 37.24729227104881, 'paviau_MS_SSIM': '', 'paviau_SAM': 3.5999231338500977, 'salinas_PSNR': 40.589812187226784, 'salinas_MS_SSIM': '', 'salinas_SAM': 1.3357473611831665, 'jasperridge_PSNR': 40.50052462795137, 'jasperridge_MS_SSIM': '', 'jasperridge_SAM': 2.1256518363952637, 'urban_PSNR': 33.8457494637348, 'urban_MS_SSIM': '', 'urban_SAM': 3.918534994125366, 'Avg_PSNR_4img': 38.04584463749044, 'Avg_MS_SSIM_4img': '', 'Avg_SAM_4img': 2.7449643313884735, 'Found_Dataset_Count': 4, 'Found_Datasets': 'paviau,salinas,jasperridge,urban'}
{'Checkpoint': 'rank8_lora3_a0.1_g1_pts5000_prune500_add5000